In [ ]:
#!/usr/bin/env python3
"""
Prepare BMMC RNA-ATAC data in the same spirit as PBMC preprocessing/split pipeline.

Outputs
-------
Base files:
- RNA_counts_qc.h5ad
- ATAC_counts_qc.h5ad
- ATAC_gas.h5ad
- feature_aligned_rna_atac.h5ad

Split files per ratio:
- results_ratio_loop_rna_atac/single_000/
- ...
Each ratio contains:
- train_rna_ref.h5ad
- train_atac_full.h5ad
- train_atac_activity.h5ad
- val_query_atac.h5ad
- val_true_rna.h5ad
- val_atac_activity.h5ad
- split_info.json
"""

from __future__ import annotations

import json
import os
import random
from pathlib import Path
from typing import List, Sequence, Tuple
from muon import atac as ac
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import episcanpy as epi
from scipy import sparse
from sklearn.model_selection import train_test_split


# =========================
# 0. Parameters
# =========================
INPUT_H5AD = "/data5/zhangye/scMRDR/input/BMMC/raw_input/GSE194122_openproblems_neurips2021_multiome_BMMC_processed.h5ad"
GTF_PATH = "/data5/zhangye/scMRDR/input/BMMC/raw_input/gencode.v38.primary_assembly.annotation.gtf"
OUTPUT_DIR = "/data5/zhangye/scMRDR/input/BMMC/preprocessed_input/RNA_ATAC"

SEED = 1234
VAL_FRAC = 0.2
SINGLE_FRACS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
RNA_KEEP_PROB = 0.5

BATCH_KEY = "batch"
MT_PREFIX = "MT-"

RNA_MIN_GENES = 200
RNA_MAX_GENES = 5000
RNA_MAX_COUNTS = 15000
RNA_MAX_MT = 20
RNA_MIN_CELLS_PER_GENE = 3

ATAC_MIN_CELLS_PER_PEAK = 10
ATAC_MIN_FEATURES_PER_CELL = 2000
ATAC_MAX_FEATURES_PER_CELL = 15000


MIN_TRAIN_PAIRED_CELLS = 50
MIN_VAL_QUERY_CELLS = 20
MIN_VAL_QUERY_CELLS = 20
MIN_COMMON_FEATURES = 50


# MIN_TRAIN_RNA_CELLS = 50
# MIN_TRAIN_PAIRED_CELLS = 50
# MIN_VAL_QUERY_CELLS = 20


# =========================
# 1. Utilities
# =========================
def set_seed(seed: int = 1234) -> None:
    random.seed(seed)
    np.random.seed(seed)


def ensure_dir(path: os.PathLike) -> None:
    Path(path).mkdir(parents=True, exist_ok=True)


def safe_write_h5ad(adata: ad.AnnData, path: os.PathLike) -> None:
    adata = adata.copy()
    if isinstance(adata.X, np.matrix):
        adata.X = np.asarray(adata.X)
    for k in list(adata.layers.keys()):
        if isinstance(adata.layers[k], np.matrix):
            adata.layers[k] = np.asarray(adata.layers[k])
    adata.write(str(path))


def to_dense(x):
    if sparse.issparse(x):
        return x.toarray()
    return np.asarray(x)


def require_obs_column(adata: ad.AnnData, key: str, fill_value: str = "batch0") -> None:
    if key not in adata.obs.columns:
        adata.obs[key] = fill_value


def add_counts_layer_if_missing(adata: ad.AnnData) -> None:
    if "counts" not in adata.layers:
        adata.layers["counts"] = adata.X.copy()


def get_common_names(*arrays: Sequence[str]) -> List[str]:
    if len(arrays) == 0:
        return []
    common = set(arrays[0])
    for arr in arrays[1:]:
        common &= set(arr)
    return sorted(common)



def subset_and_copy(adata: ad.AnnData, cells: Sequence[str], features: Sequence[str] | None = None) -> ad.AnnData:
    out = adata[list(cells)].copy()
    if features is not None:
        out = out[:, list(features)].copy()
    return out


def assign_partial_modality(cells: Sequence[str], single_frac: float = 0.2, rna_keep_prob: float = 0.5):
    cells = list(cells)
    n = len(cells)
    n_single = round(n * single_frac)

    single_cells = random.sample(cells, n_single) if n_single > 0 else []
    paired_cells = sorted(set(cells) - set(single_cells))

    modality = {c: "paired" for c in cells}
    if n_single > 0:
        for c in single_cells:
            modality[c] = "RNA" if random.random() < rna_keep_prob else "ATAC"

    rna_only = sorted([c for c, m in modality.items() if m == "RNA"])
    atac_only = sorted([c for c, m in modality.items() if m == "ATAC"])

    return {
        "modality": modality,
        "paired_cells": paired_cells,
        "rna_only_cells": rna_only,
        "atac_only_cells": atac_only,
    }


def save_split_h5ads(
    outdir: os.PathLike,
    train_rna_ref: ad.AnnData,
    train_atac_full: ad.AnnData,
    train_atac_activity: ad.AnnData,
    val_query_atac: ad.AnnData,
    val_true_rna: ad.AnnData,
    val_atac_activity: ad.AnnData,
) -> None:
    outdir = Path(outdir)
    safe_write_h5ad(train_rna_ref, outdir / "train_rna_ref.h5ad")
    safe_write_h5ad(train_atac_full, outdir / "train_atac_full.h5ad")
    safe_write_h5ad(train_atac_activity, outdir / "train_atac_activity.h5ad")
    safe_write_h5ad(val_query_atac, outdir / "val_query_atac.h5ad")
    safe_write_h5ad(val_true_rna, outdir / "val_true_rna.h5ad")
    safe_write_h5ad(val_atac_activity, outdir / "val_atac_activity.h5ad")


# =========================
# 2. Read and preprocess
# =========================
def read_bmmc_multiome(input_h5ad: os.PathLike) -> Tuple[ad.AnnData, ad.AnnData]:
    adata = sc.read_h5ad(str(input_h5ad))
    if "counts" in adata.layers:
        adata.X = adata.layers["counts"].copy()

    if "feature_types" not in adata.var.columns:
        raise ValueError("feature_types not found in adata.var")

    rna = adata[:, adata.var["feature_types"].astype(str) == "GEX"].copy()
    atac = adata[:, adata.var["feature_types"].astype(str) == "ATAC"].copy()

    # keep only modality-relevant obsm
    rna.obsm = {k: rna.obsm[k] for k in ["GEX_X_pca", "GEX_X_umap"] if k in rna.obsm}
    atac.obsm = {k: atac.obsm[k] for k in ["ATAC_gene_activity", "ATAC_lsi_full", "ATAC_lsi_red", "ATAC_umap"] if k in atac.obsm}
    return rna, atac


def preprocess_rna(rna: ad.AnnData) -> ad.AnnData:
    rna = rna.copy()
    require_obs_column(rna, BATCH_KEY, "batch0")

    rna.var["mt"] = rna.var_names.str.startswith(MT_PREFIX)
    sc.pp.calculate_qc_metrics(rna, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)

    sc.pp.filter_genes(rna, min_cells=RNA_MIN_CELLS_PER_GENE)
    sc.pp.filter_cells(rna, min_genes=RNA_MIN_GENES)
    rna = rna[rna.obs["n_genes_by_counts"] <= RNA_MAX_GENES, :].copy()
    rna = rna[rna.obs["total_counts"] <= RNA_MAX_COUNTS, :].copy()
    rna = rna[rna.obs["pct_counts_mt"] < RNA_MAX_MT, :].copy()

    rna.layers["counts"] = rna.X.copy()
    sc.pp.normalize_total(rna)
    sc.pp.log1p(rna)
    sc.pp.highly_variable_genes(rna, batch_key=BATCH_KEY, min_mean=0.02, max_mean=4, min_disp=0.5)
    return rna


def preprocess_atac(atac: ad.AnnData) -> ad.AnnData:
    atac = atac.copy()
    require_obs_column(atac, BATCH_KEY, "batch0")

    sc.pp.calculate_qc_metrics(atac, percent_top=None, log1p=False, inplace=True)
    sc.pp.filter_genes(atac, min_cells=ATAC_MIN_CELLS_PER_PEAK)
    sc.pp.filter_cells(atac, min_genes=ATAC_MIN_FEATURES_PER_CELL)
    atac = atac[atac.obs["n_genes_by_counts"] <= ATAC_MAX_FEATURES_PER_CELL, :].copy()

    atac.layers["counts"] = atac.X.copy()
    ac.pp.tfidf(atac, scale_factor=1e4)
    sc.pp.highly_variable_genes(atac, min_mean=0.05, max_mean=1.5, min_disp=0.5, batch_key=BATCH_KEY)

    # geneactivity in episcanpy prefers chr:start-end style
    atac.var_names = [name.replace("-", ":", 1) if "-" in name and ":" not in name else name for name in atac.var_names]
    return atac


def build_atac_gene_activity(atac_qc: ad.AnnData, gtf_path: os.PathLike) -> ad.AnnData:
    atac = atac_qc.copy()
    atac.X = atac.layers["counts"].copy()
    gas = epi.tl.geneactivity(atac, str(gtf_path), annotation="HAVANA")
    gas = gas[:, ~gas.var_names.duplicated()].copy()
    gas.layers["counts"] = gas.X.copy()

    # Optional normalization for downstream transfer-style usage
    try:
        import muon.atac as ac
        ac.pp.tfidf(gas, scale_factor=1e4)
    except Exception:
        pass

    sc.pp.highly_variable_genes(gas)
    return gas


def build_feature_aligned(rna_qc: ad.AnnData, atac_gas: ad.AnnData) -> ad.AnnData:
    genelist = list(rna_qc.var_names[rna_qc.var["highly_variable"]].astype(str))
    gas_hvg = list(atac_gas.var_names[atac_gas.var["highly_variable"]].astype(str))
    union_genes = sorted(set(genelist) | set(gas_hvg))

    rna = rna_qc.copy()
    gas = atac_gas.copy()
    adata = ad.concat([rna, gas], join="outer", label="modality", keys=["rna", "atac_activity"])
    adata.uns["rna_hvg"] = genelist
    adata.uns["atac_hvg"] = gas_hvg
    adata.uns["rna_nz"] = sorted(set(union_genes) & set(rna.var_names.astype(str)))
    adata.uns["atac_nz"] = sorted(set(union_genes) & set(gas.var_names.astype(str)))
    adata = adata[:, union_genes].copy()
    return adata


# =========================
# 3. Split generation
# =========================

def generate_splits(
    rna_qc_path: os.PathLike,
    atac_qc_path: os.PathLike,
    atac_gas_path: os.PathLike,
    out_root: os.PathLike,
    seed: int = 1234,
    val_frac: float = 0.2,
    single_fracs: Sequence[float] = (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
    rna_keep_prob: float = 0.5,
    min_train_rna_cells: int = 50,
    min_val_query_cells: int = 20,
    min_common_features: int = 50,
) -> pd.DataFrame:
    rna = sc.read_h5ad(str(rna_qc_path))
    atac_peak = sc.read_h5ad(str(atac_qc_path))
    atac_gas = sc.read_h5ad(str(atac_gas_path))

    # 细胞交集：三者都要有
    common_cells = get_common_names(
        rna.obs_names.tolist(),
        atac_peak.obs_names.tolist(),
        atac_gas.obs_names.tolist()
    )
    if len(common_cells) == 0:
        raise ValueError("No common cells shared by RNA_counts_qc, ATAC_counts_qc, and ATAC_gas.")

    rna = rna[common_cells].copy()
    atac_peak = atac_peak[common_cells].copy()
    atac_gas = atac_gas[common_cells].copy()

    # RNA 与 gene activity 的共同基因特征
    common_features = get_common_names(
        rna.var_names.tolist(),
        atac_gas.var_names.tolist()
    )
    if len(common_features) < min_common_features:
        raise ValueError(
            f"Too few common gene-level features between RNA and ATAC_gas: {len(common_features)}"
        )

    rna = rna[:, common_features].copy()
    atac_gas = atac_gas[:, common_features].copy()

    all_cells = np.array(common_cells, dtype=object)
    train_cells, val_cells = train_test_split(
        all_cells,
        test_size=val_frac,
        random_state=seed,
        shuffle=True
    )
    train_cells = np.array(sorted(train_cells.tolist()), dtype=object)
    val_cells = np.array(sorted(val_cells.tolist()), dtype=object)

    ensure_dir(out_root)
    summaries = []

    for sf in single_fracs:
        ratio_label = f"single_{int(round(sf * 100)):03d}"
        outdir = Path(out_root) / ratio_label
        ensure_dir(outdir)

        train_assign = assign_partial_modality(
            train_cells,
            single_frac=sf,
            rna_keep_prob=rna_keep_prob,
            # seed=seed + int(round(sf * 1000)) + 11
        )
        val_assign = assign_partial_modality(
            val_cells,
            single_frac=sf,
            rna_keep_prob=rna_keep_prob,
            # seed=seed + int(round(sf * 1000)) + 97
        )

        # 和 R 逻辑保持一致
        train_rna_ref_cells = sorted(train_assign["paired_cells"] + train_assign["rna_only_cells"])
        train_atac_ref_cells = sorted(train_assign["paired_cells"] + train_assign["atac_only_cells"])
        val_query_atac_cells = sorted(val_assign["paired_cells"] + val_assign["atac_only_cells"])

        if (
            len(train_rna_ref_cells) < min_train_rna_cells
            or len(train_atac_ref_cells) < min_train_rna_cells
            or len(val_query_atac_cells) < min_val_query_cells
        ):
            meta = {
                "ratio_label": ratio_label,
                "single_frac": sf,
                "status": "skipped",
                "reason": "too_few_cells",
                "train_rna_ref": len(train_rna_ref_cells),
                "train_atac_ref": len(train_atac_ref_cells),
                "val_query_atac": len(val_query_atac_cells),
            }
            with open(outdir / "split_info.json", "w", encoding="utf-8") as f:
                json.dump(meta, f, indent=2, ensure_ascii=False)
            summaries.append(meta)
            continue

        # -------- 保存各类 split 文件 --------
        train_rna_ref = subset_and_copy(rna, train_rna_ref_cells)

        # train ATAC full: 保留整套训练 ATAC peak-level 数据（沿用你原来的命名）
        train_atac_full = subset_and_copy(atac_peak, train_cells)

        # 新增：train ATAC activity，只取 train ATAC reference cells
        train_atac_activity = subset_and_copy(atac_gas, train_atac_ref_cells)

        # val query peak-level ATAC
        val_query_atac = subset_and_copy(atac_peak, val_query_atac_cells)

        # val true RNA
        val_true_rna = subset_and_copy(rna, val_query_atac_cells)

        # val ATAC activity
        val_atac_activity = subset_and_copy(atac_gas, val_query_atac_cells)
        
        

        save_split_h5ads(
            outdir,
            train_rna_ref=train_rna_ref,
            train_atac_full=train_atac_full,
            train_atac_activity=train_atac_activity,
            val_query_atac=val_query_atac,
            val_true_rna=val_true_rna,
            val_atac_activity=val_atac_activity,
        )

        meta = {
            "ratio_label": ratio_label,
            "single_frac": sf,
            "status": "ok",
            "train_cells": train_cells.tolist(),
            "val_cells": val_cells.tolist(),
            "train_paired_cells": train_assign["paired_cells"],
            "train_rna_only_cells": train_assign["rna_only_cells"],
            "train_atac_only_cells": train_assign["atac_only_cells"],
            "val_paired_cells": val_assign["paired_cells"],
            "val_rna_only_cells": val_assign["rna_only_cells"],
            "val_atac_only_cells": val_assign["atac_only_cells"],
            "train_rna_ref_cells": train_rna_ref_cells,
            "train_atac_ref_cells": train_atac_ref_cells,
            "val_query_atac_cells": val_query_atac_cells,
            "common_gene_features": common_features,
            "n_train_rna_ref_cells": len(train_rna_ref_cells),
            "n_train_atac_ref_cells": len(train_atac_ref_cells),
            "n_val_query_atac_cells": len(val_query_atac_cells),
            "n_common_features": len(common_features),
        }
        with open(outdir / "split_info.json", "w", encoding="utf-8") as f:
            json.dump(meta, f, indent=2, ensure_ascii=False)

        summaries.append({
            "ratio_label": ratio_label,
            "single_frac": sf,
            "status": "ok",
            "train_paired": len(train_assign["paired_cells"]),
            "train_rna_only": len(train_assign["rna_only_cells"]),
            "train_atac_only": len(train_assign["atac_only_cells"]),
            "val_paired": len(val_assign["paired_cells"]),
            "val_rna_only": len(val_assign["rna_only_cells"]),
            "val_atac_only": len(val_assign["atac_only_cells"]),
            "train_rna_ref": len(train_rna_ref_cells),
            "train_atac_ref": len(train_atac_ref_cells),
            "val_query_atac": len(val_query_atac_cells),
            "n_common_features": len(common_features),
        })

    summary_df = pd.DataFrame(summaries)
    summary_df.to_csv(Path(out_root) / "summary_all_ratios.csv", index=False)
    return summary_df



# =========================
# 4. Main
# =========================
def main():
    set_seed(SEED)
    ensure_dir(OUTPUT_DIR)

    outdir = Path(OUTPUT_DIR)
    rna_qc_path = outdir / "RNA_counts_qc.h5ad"
    atac_qc_path = outdir / "ATAC_counts_qc.h5ad"
    atac_gas_path = outdir / "ATAC_gas.h5ad"
    feature_aligned_path = outdir / "feature_aligned_rna_atac.h5ad"

    print("Reading BMMC multiome...")
    rna_raw, atac_raw = read_bmmc_multiome(INPUT_H5AD)
    print("RNA raw:", rna_raw.shape)
    print("ATAC raw:", atac_raw.shape)

    # print("[1/4] Preprocess RNA")
    # rna_qc = preprocess_rna(rna_raw)
    # safe_write_h5ad(rna_qc, rna_qc_path)

    # print("[2/4] Preprocess ATAC")
    # atac_qc = preprocess_atac(atac_raw)
    # safe_write_h5ad(atac_qc, atac_qc_path)

    # print("[3/4] Build ATAC gene activity")
    # atac_gas = build_atac_gene_activity(atac_qc, GTF_PATH)
    # safe_write_h5ad(atac_gas, atac_gas_path)

    # print("[4/4] Build feature-aligned file")
    # feature_aligned = build_feature_aligned(rna_qc, atac_gas)
    # safe_write_h5ad(feature_aligned, feature_aligned_path)

    print("Generating ratio splits...")
    summary_df = generate_splits(
        rna_qc_path=rna_qc_path,
        atac_qc_path=atac_qc_path,
        atac_gas_path=atac_gas_path,
        out_root=outdir / "results_ratio_loop_rna_atac",
        seed=SEED,
        val_frac=VAL_FRAC,
        single_fracs=SINGLE_FRACS,
        rna_keep_prob=RNA_KEEP_PROB,
    )
    print(summary_df)


if __name__ == "__main__":
    main()



Reading BMMC multiome...
RNA raw: (69249, 13431)
ATAC raw: (69249, 116490)
Generating ratio splits...
  ratio_label  single_frac status  train_paired  train_rna_only  \
0  single_000          0.0     ok         41257               0   
1  single_020          0.2     ok         33006            4097   
2  single_040          0.4     ok         24754            8174   
3  single_060          0.6     ok         16503           12206   
4  single_080          0.8     ok          8251           16614   
5  single_100          1.0     ok             0           20817   

   train_atac_only  val_paired  val_rna_only  val_atac_only  train_rna_ref  \
0                0       10315             0              0          41257   
1             4154        8252           996           1067          37103   
2             8329        6189          2104           2022          32928   
3            12548        4126          3141           3048          28709   
4            16392        2063        